# DistributedDataParallel 基础

## 学习目标

理解 rank、world size、DistributedSampler 和 DDP 的职责，并在单进程环境完成初始化、训练一步和清理。

## 概念模型

DDP 为每个进程复制模型，每个进程处理不同数据分片，反向传播时自动同步梯度。生产多卡训练还需要 torchrun、多进程启动和只由 rank 0 保存日志/checkpoint。

In [ ]:
import os, tempfile
import torch
import torch.distributed as dist
from torch import nn
from torch.nn.parallel import DistributedDataParallel as DDP

init_file = tempfile.NamedTemporaryFile(delete=False).name
try:
    dist.init_process_group('gloo', init_method=f'file://{init_file}', rank=0, world_size=1)
    model = DDP(nn.Linear(4, 2))
    x = torch.randn(8, 4); y = torch.randint(0, 2, (8,))
    optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
    loss = nn.CrossEntropyLoss()(model(x), y)
    loss.backward(); optimizer.step()
    print('rank/world:', dist.get_rank(), dist.get_world_size(), 'loss:', loss.item())
except RuntimeError as error:
    print('DDP runtime unavailable in this environment:', error)
finally:
    if dist.is_initialized(): dist.destroy_process_group()
    if os.path.exists(init_file): os.unlink(init_file)

### 实验 1：使用 DistributedSampler 做数据分片

**实验目的**：在单 rank 冒烟环境中展示 DistributedSampler 的接口，并调用 `set_epoch()` 让每个 epoch 的 shuffle 使用不同且各 rank 一致的种子。

真实 DDP 中每个进程拥有独立模型副本和数据分片，反向时同步梯度。数据量不能整除 world size 时 sampler 可能补齐或丢弃样本；指标汇总需要跨 rank reduce。


In [ ]:
from torch.utils.data import TensorDataset, DataLoader
dataset = TensorDataset(torch.arange(10).float().unsqueeze(1), torch.zeros(10, dtype=torch.long))
sampler = torch.utils.data.distributed.DistributedSampler(dataset, num_replicas=1, rank=0, shuffle=True)
sampler.set_epoch(0)
loader = DataLoader(dataset, batch_size=2, sampler=sampler)
print('distributed batches:', [batch[0].flatten().tolist() for batch in loader])
assert sum(len(batch[0]) for batch in loader) == len(dataset)

## 官方教程补充

**对应官方源文件：** `beginner_source/ddp_series_intro.rst`、`beginner_source/ddp_series_theory.rst`、`intermediate_source/ddp_tutorial.rst`、`beginner_source/dist_overview.rst`

官方 DDP 采用每进程一份模型：各 rank 处理不同数据分片，反向时对梯度 all-reduce，使参数更新保持一致。`DistributedSampler` 需要每 epoch 调用 `set_epoch` 以获得一致但变化的 shuffle；日志、评估聚合和 checkpoint 通常只由 rank 0 写入。启动、设备绑定和进程组销毁都属于生命周期契约，单卡正确不代表多卡无死锁。

**验证练习：** 找到上面源文件中的对应 API，先写出输入、输出和状态变化，再运行本 notebook 的相关实验；如果行为不同，优先检查本地 PyTorch 版本、设备能力和输入契约。

<!-- official-pytorch-supplement-v1 -->

## 检查点

解释 rank、world size、sampler 和梯度同步的关系；说明为什么普通 shuffle 不能替代 DistributedSampler。

## 试一试

把 world size 改成 2，使用 `torchrun --nproc-per-node=2` 启动一个脚本，并只让 rank 0 保存 checkpoint。

## 常见错误与调试

忘记初始化进程组、每个进程重复读取全部数据、遗漏 `set_epoch()`、所有进程同时写同一个文件、没有 destroy process group。